
# Projeto 13 — Taxa de Homicídios por 100 mil Habitantes por UF (2010–2024)

**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Objetivo do projeto:** Analisar variações regionais e temporais na taxa de homicídios por 100 mil habitantes, aplicando os conceitos trabalhados nas aulas 1 a 6.

---
## Estrutura do notebook
1. Título e introdução  
2. Definição do problema  
3. Definição do KPI  
4. Aquisição dos dados  
5. Limpeza e preparação dos dados  
6. Análise exploratória  
7. Visualizações  
8. Insights e interpretação  
9. Conclusão  
10. Reflexão final (data storytelling)



## 1. Introdução

A taxa de homicídios por 100 mil habitantes é um indicador amplamente utilizado para comparar níveis de violência entre diferentes unidades da federação, pois corrige o efeito do tamanho populacional. Essa análise permite identificar padrões temporais, desigualdades regionais e estados mais críticos.

Neste projeto, vamos investigar como a taxa de homicídios evoluiu entre 2010 e 2024 e quais UFs apresentaram os maiores e os menores valores ao longo do período.



## 2. Definição do problema

**Pergunta de análise:**  
Como a taxa de homicídios por 100 mil habitantes evoluiu entre 2010 e 2024 nas unidades da federação e quais estados apresentam os maiores níveis de violência?

**Por que essa pergunta é relevante?**  
Porque permite comparar a violência de forma proporcional, identificando áreas críticas e tendências temporais que podem apoiar políticas públicas de segurança.

**Qual decisão poderia ser tomada com base nessa análise?**  
Priorizar estados com maior taxa de homicídios em estratégias de prevenção, segurança pública e monitoramento.



## 3. Definição do KPI

### KPI principal
**Taxa de homicídios por 100 mil habitantes**

### Fórmula
\[
	ext{Taxa de Homicídios} = rac{	ext{Número de Homicídios}}{	ext{População}} 	imes 100000
\]

### Justificativa
Esse indicador permite comparar diferentes estados de forma justa, sem que o tamanho da população distorça os resultados.

### KPI complementar
- Média da taxa por UF no período
- Variação percentual ao longo do tempo
- Ano com maior taxa média nacional


In [ ]:

# 4. Aquisição dos dados

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_theme(style="whitegrid")

# Leitura da base
df = pd.read_excel("Simulacao_Taxa_Homicidios_UF_2010_2024.xlsx")

# Visualização inicial
df.head()



### Descrição das variáveis principais
- **Ano**: ano de referência
- **UF**: unidade da federação
- **Homicídios**: número total de homicídios registrados
- **População**: população estimada da UF
- **Taxa por 100 mil habitantes**: indicador principal da análise


In [ ]:

# 5. Limpeza e preparação dos dados

print("Dimensões da base:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nQuantidade de linhas duplicadas:", df.duplicated().sum())


In [ ]:

# Ajuste de nomes de colunas (defensivo)
rename_map = {
    "Taxa por 100 mil habitantes": "Taxa_Homicidios",
    "Taxa de Homicídios por 100 mil": "Taxa_Homicidios",
    "Taxa Homicídios": "Taxa_Homicidios",
    "Homicidios": "Homicidios",
    "Populacao": "Populacao"
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# Se a base não vier com a taxa pronta, calcular a taxa
if "Taxa_Homicidios" not in df.columns and {"Homicidios", "Populacao"}.issubset(df.columns):
    df["Taxa_Homicidios"] = (df["Homicidios"] / df["Populacao"]) * 100000

# Garantindo tipos coerentes
df["Ano"] = df["Ano"].astype(int)
df["UF"] = df["UF"].astype(str)
df["Taxa_Homicidios"] = df["Taxa_Homicidios"].astype(float)

df.head()



### Comentário sobre a preparação
Nesta etapa:
- verificamos a estrutura da base;
- ajustamos nomes de colunas quando necessário;
- garantimos os tipos corretos;
- calculamos a taxa por 100 mil habitantes, caso ela não estivesse pronta na base.


In [ ]:

# 6. Análise exploratória

# Estatísticas descritivas da taxa
df["Taxa_Homicidios"].describe()


In [ ]:

# Média da taxa por ano (visão nacional agregada)
taxa_por_ano = df.groupby("Ano")["Taxa_Homicidios"].mean().reset_index()
taxa_por_ano


In [ ]:

# Média da taxa por UF no período
taxa_por_uf = (
    df.groupby("UF")["Taxa_Homicidios"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
taxa_por_uf


In [ ]:

# Tabela dinâmica: taxa por ano e UF
taxa_uf_ano = df.pivot_table(
    index="Ano",
    columns="UF",
    values="Taxa_Homicidios",
    aggfunc="mean"
)

taxa_uf_ano


In [ ]:

# Variação percentual da taxa por UF
df = df.sort_values(["UF", "Ano"])
df["Variacao_Percentual"] = df.groupby("UF")["Taxa_Homicidios"].pct_change() * 100

df.head(10)



## 7. Visualizações

Os gráficos abaixo ajudam a responder:
- quais estados apresentam maiores taxas;
- como a violência evolui ao longo do tempo;
- quais períodos são mais críticos.


In [ ]:

# Gráfico 1: evolução da taxa média nacional ao longo do tempo
plt.figure(figsize=(10, 6))
sns.lineplot(data=taxa_por_ano, x="Ano", y="Taxa_Homicidios", marker="o")
plt.title("Evolução da Taxa Média de Homicídios por 100 mil Habitantes (2010–2024)")
plt.xlabel("Ano")
plt.ylabel("Taxa de Homicídios")
plt.xticks(taxa_por_ano["Ano"], rotation=45)
plt.show()


In [ ]:

# Gráfico 2: ranking de UFs por taxa média no período
plt.figure(figsize=(10, 6))
sns.barplot(data=taxa_por_uf, x="UF", y="Taxa_Homicidios")
plt.title("Taxa Média de Homicídios por UF (2010–2024)")
plt.xlabel("UF")
plt.ylabel("Taxa de Homicídios")
plt.show()


In [ ]:

# Gráfico 3: evolução da taxa por UF
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x="Ano", y="Taxa_Homicidios", hue="UF", legend=False)
plt.title("Evolução da Taxa de Homicídios por UF")
plt.xlabel("Ano")
plt.ylabel("Taxa de Homicídios")
plt.xticks(sorted(df["Ano"].unique()), rotation=45)
plt.show()


In [ ]:

# Gráfico 4: distribuição das taxas
plt.figure(figsize=(8, 5))
sns.histplot(df["Taxa_Homicidios"], bins=12, kde=True)
plt.title("Distribuição da Taxa de Homicídios")
plt.xlabel("Taxa de Homicídios")
plt.ylabel("Frequência")
plt.show()



## 8. Insights e interpretação

Nesta etapa, a análise deve explicar os padrões observados, e não apenas descrever os gráficos.


In [ ]:

# UF com maior taxa média no período
uf_mais_critica = taxa_por_uf.loc[taxa_por_uf["Taxa_Homicidios"].idxmax()]
uf_mais_critica


In [ ]:

# UF com menor taxa média no período
uf_menos_critica = taxa_por_uf.loc[taxa_por_uf["Taxa_Homicidios"].idxmin()]
uf_menos_critica


In [ ]:

# Ano com maior taxa média nacional
ano_mais_critico = taxa_por_ano.loc[taxa_por_ano["Taxa_Homicidios"].idxmax()]
ano_mais_critico



### Modelo de interpretação esperada

**Insight 1 — Diferenças entre estados**  
A comparação entre as UFs mostra que a violência não se distribui de forma homogênea no país. Algumas unidades apresentam taxas consistentemente mais elevadas, o que indica necessidade de maior atenção em políticas de segurança pública.

**Insight 2 — Evolução temporal**  
A série histórica permite verificar se a taxa média de homicídios está aumentando, diminuindo ou permanecendo estável ao longo do tempo. Mudanças importantes podem refletir alterações em contexto social, políticas públicas ou dinâmicas criminais.

**Insight 3 — Regiões críticas**  
Os estados com maiores taxas devem ser priorizados em ações estratégicas de prevenção, monitoramento e intervenção.

**Insight 4 — Limitações da análise**  
A base mostra o comportamento da taxa ao longo do tempo, mas não explica sozinha os motivos dessas variações. Para uma análise mais aprofundada, seria importante incorporar dados socioeconômicos, políticas de segurança e desigualdade social.



## 9. Conclusão

Com base na análise realizada, é possível identificar quais unidades da federação apresentam as maiores taxas médias de homicídio e como esse comportamento evoluiu entre 2010 e 2024.

Esses resultados ajudam a mapear estados mais críticos e períodos que exigem maior atenção, apoiando decisões voltadas à segurança pública e à prevenção da violência.

**Resposta à pergunta inicial:**  
A taxa de homicídios por 100 mil habitantes variou entre os estados e ao longo do tempo, com algumas UFs se destacando negativamente por apresentarem níveis mais elevados de violência.



## 10. Reflexão Final (Data Storytelling)

Se este projeto fosse apresentado a um gestor, a principal mensagem seria:

> “A violência letal não afeta todas as unidades da federação da mesma forma. Há estados com taxas persistentemente mais elevadas, que precisam ser priorizados em políticas públicas de segurança e prevenção.”

### Ação recomendada
- Direcionar recursos para os estados com maior taxa;
- Investigar padrões regionais e fatores associados;
- Complementar a análise com dados sociais e econômicos em estudos futuros.



## Sugestões de aprofundamento (opcional)
- Comparar médias por região do Brasil;
- Analisar ranking anual de UFs;
- Destacar estados com maior redução ou crescimento da taxa;
- Transformar a análise em dashboard interativo em etapa futura.
